In [3]:
!pip -q install yfinance

In [4]:
# Google Colab script for ONE ticker using yfinance
# - Fetches price/fundamental/financial data from yfinance
# - Fetches news from SerpAPI
# - Creates a simple sentiment proxy from headlines/snippets
# - Prints structured output
# - Prints a ready-to-paste AI prompt for analysis/recommendation
#
# Required Colab secrets:
#   SERPAPI_API_KEY
#
# Example tickers:
#   US: AAPL, MSFT, NVDA, GRAB
#   SGX via Yahoo format may vary; often Singapore tickers use .SI

!pip -q install yfinance

import yfinance as yf
import requests
import json
import time
import pandas as pd
from google.colab import userdata

SERPAPI_KEY = userdata.get("SERPAPI_KEY")
if not SERPAPI_KEY:
    raise ValueError("Missing Colab secret: SERPAPI_API_KEY")

# =========================
# CONFIG
# =========================
TICKER = "YOURTICKERHERE"
SERPAPI_BASE = "https://serpapi.com/search.json"
SLEEP_BETWEEN_SERP_CALLS = 2


# =========================
# HELPERS
# =========================
def safe_get(url, params, timeout=30):
    try:
        r = requests.get(url, params=params, timeout=timeout)
        r.raise_for_status()
        return r.json()
    except Exception as e:
        return {"_error": str(e)}

def parse_float(x):
    try:
        if x is None or str(x) in ["", "None", "nan", "NaN", "N/A", "-"]:
            return None
        return float(x)
    except:
        return None

def parse_int(x):
    try:
        if x is None or str(x) in ["", "None", "nan", "NaN", "N/A", "-"]:
            return None
        return int(float(x))
    except:
        return None

def safe_value(d, key, default=None):
    if isinstance(d, dict):
        return d.get(key, default)
    return default

def is_sgx_ticker(ticker):
    return ticker.upper().endswith(".SI")

def company_query_name(ticker, info=None):
    if isinstance(info, dict):
        name = info.get("longName") or info.get("shortName")
        if name:
            return f"{name} stock"
    return f"{ticker} stock"

def df_value(df, row_name, col_idx=0):
    try:
        if df is None or df.empty:
            return None
        if row_name not in df.index:
            return None
        cols = list(df.columns)
        if not cols:
            return None
        col = cols[col_idx]
        return df.loc[row_name, col]
    except:
        return None


# =========================
# YFINANCE FETCH
# =========================
def get_yf_ticker(ticker):
    return yf.Ticker(ticker)

def get_info(tkr):
    try:
        return tkr.info
    except Exception as e:
        return {"_error": str(e)}

def get_price_history(tkr, period="6mo", interval="1d"):
    try:
        hist = tkr.history(period=period, interval=interval, auto_adjust=False)
        return hist
    except Exception as e:
        return pd.DataFrame()

def get_income_statement(tkr):
    try:
        return tkr.financials
    except:
        return pd.DataFrame()

def get_balance_sheet(tkr):
    try:
        return tkr.balance_sheet
    except:
        return pd.DataFrame()

def get_cash_flow(tkr):
    try:
        return tkr.cashflow
    except:
        return pd.DataFrame()

def get_quarterly_income_statement(tkr):
    try:
        return tkr.quarterly_financials
    except:
        return pd.DataFrame()

def get_quarterly_balance_sheet(tkr):
    try:
        return tkr.quarterly_balance_sheet
    except:
        return pd.DataFrame()

def get_quarterly_cash_flow(tkr):
    try:
        return tkr.quarterly_cashflow
    except:
        return pd.DataFrame()

def get_earnings_dates(tkr):
    try:
        return tkr.earnings_dates
    except:
        return pd.DataFrame()


# =========================
# SUMMARIZERS
# =========================
def summarize_overview(info):
    if not isinstance(info, dict) or len(info) == 0:
        return {"error": "No overview data found"}

    return {
        "symbol": info.get("symbol"),
        "name": info.get("longName") or info.get("shortName"),
        "exchange": info.get("exchange"),
        "currency": info.get("currency"),
        "country": info.get("country"),
        "sector": info.get("sector"),
        "industry": info.get("industry"),
        "market_cap": parse_int(info.get("marketCap")),
        "enterprise_value": parse_int(info.get("enterpriseValue")),
        "trailing_pe": parse_float(info.get("trailingPE")),
        "forward_pe": parse_float(info.get("forwardPE")),
        "peg_ratio": parse_float(info.get("pegRatio")),
        "price_to_book": parse_float(info.get("priceToBook")),
        "book_value": parse_float(info.get("bookValue")),
        "dividend_rate": parse_float(info.get("dividendRate")),
        "dividend_yield": parse_float(info.get("dividendYield")),
        "eps_trailing": parse_float(info.get("trailingEps")),
        "eps_forward": parse_float(info.get("forwardEps")),
        "revenue_ttm": parse_int(info.get("totalRevenue")),
        "profit_margin": parse_float(info.get("profitMargins")),
        "operating_margin": parse_float(info.get("operatingMargins")),
        "return_on_assets": parse_float(info.get("returnOnAssets")),
        "return_on_equity": parse_float(info.get("returnOnEquity")),
        "target_mean_price": parse_float(info.get("targetMeanPrice")),
        "beta": parse_float(info.get("beta")),
        "week_52_high": parse_float(info.get("fiftyTwoWeekHigh")),
        "week_52_low": parse_float(info.get("fiftyTwoWeekLow")),
        "business_summary": info.get("longBusinessSummary")
    }

def summarize_price_data(hist):
    if hist is None or hist.empty:
        return {"error": "No price history found"}

    hist = hist.copy().dropna()
    if hist.empty:
        return {"error": "Price history is empty after dropping NA"}

    latest = hist.iloc[-1]

    result = {
        "latest_trading_day": str(hist.index[-1].date()),
        "open": parse_float(latest.get("Open")),
        "high": parse_float(latest.get("High")),
        "low": parse_float(latest.get("Low")),
        "close": parse_float(latest.get("Close")),
        "volume": parse_int(latest.get("Volume"))
    }

    close = result["close"]

    if len(hist) > 1:
        prev_close = parse_float(hist.iloc[-2].get("Close"))
        if prev_close and close:
            result["change_1d_pct"] = round((close / prev_close - 1) * 100, 2)

    if len(hist) > 5:
        p5 = parse_float(hist.iloc[-6].get("Close"))
        if p5 and close:
            result["change_5d_pct"] = round((close / p5 - 1) * 100, 2)

    if len(hist) > 21:
        p21 = parse_float(hist.iloc[-22].get("Close"))
        if p21 and close:
            result["change_21d_pct"] = round((close / p21 - 1) * 100, 2)

    return result

def summarize_income_statement(annual_df, quarterly_df):
    out = {}

    if annual_df is not None and not annual_df.empty:
        revenue = df_value(annual_df, "Total Revenue", 0)
        op_income = df_value(annual_df, "Operating Income", 0)
        net_income = df_value(annual_df, "Net Income", 0)

        revenue_prev = df_value(annual_df, "Total Revenue", 1)
        net_income_prev = df_value(annual_df, "Net Income", 1)

        out["annual"] = {
            "period_end": str(annual_df.columns[0].date()) if len(annual_df.columns) > 0 else None,
            "revenue": parse_int(revenue),
            "operating_income": parse_int(op_income),
            "net_income": parse_int(net_income),
            "revenue_yoy_pct": round((revenue / revenue_prev - 1) * 100, 2) if revenue and revenue_prev else None,
            "net_income_yoy_pct": round((net_income / net_income_prev - 1) * 100, 2) if net_income and net_income_prev else None
        }

    if quarterly_df is not None and not quarterly_df.empty:
        q_revenue = df_value(quarterly_df, "Total Revenue", 0)
        q_net_income = df_value(quarterly_df, "Net Income", 0)
        q_revenue_prev_year = df_value(quarterly_df, "Total Revenue", 4)
        q_net_income_prev_year = df_value(quarterly_df, "Net Income", 4)

        out["quarterly"] = {
            "period_end": str(quarterly_df.columns[0].date()) if len(quarterly_df.columns) > 0 else None,
            "revenue": parse_int(q_revenue),
            "net_income": parse_int(q_net_income),
            "revenue_yoy_pct": round((q_revenue / q_revenue_prev_year - 1) * 100, 2) if q_revenue and q_revenue_prev_year else None,
            "net_income_yoy_pct": round((q_net_income / q_net_income_prev_year - 1) * 100, 2) if q_net_income and q_net_income_prev_year else None
        }

    return out if out else {"error": "No income statement data found"}

def summarize_balance_sheet(annual_df, quarterly_df):
    df = quarterly_df if quarterly_df is not None and not quarterly_df.empty else annual_df
    basis = "quarterly" if quarterly_df is not None and not quarterly_df.empty else "annual"

    if df is None or df.empty:
        return {"error": "No balance sheet data found"}

    total_assets = parse_int(df_value(df, "Total Assets", 0))
    total_liabilities = parse_int(df_value(df, "Total Liabilities Net Minority Interest", 0))
    equity = parse_int(df_value(df, "Stockholders Equity", 0))
    cash = parse_int(df_value(df, "Cash And Cash Equivalents", 0))
    current_assets = parse_int(df_value(df, "Current Assets", 0))
    current_liabilities = parse_int(df_value(df, "Current Liabilities", 0))
    long_term_debt = parse_int(df_value(df, "Long Term Debt", 0))
    current_debt = parse_int(df_value(df, "Current Debt", 0))

    total_debt = (long_term_debt or 0) + (current_debt or 0)

    return {
        "period_end": str(df.columns[0].date()) if len(df.columns) > 0 else None,
        "basis": basis,
        "total_assets": total_assets,
        "total_liabilities": total_liabilities,
        "shareholder_equity": equity,
        "cash": cash,
        "current_assets": current_assets,
        "current_liabilities": current_liabilities,
        "long_term_debt": long_term_debt,
        "current_debt": current_debt,
        "total_debt_estimate": total_debt if total_debt else None,
        "current_ratio": round(current_assets / current_liabilities, 2) if current_assets and current_liabilities else None,
        "debt_to_equity": round(total_debt / equity, 2) if total_debt and equity else None
    }

def summarize_cash_flow(annual_df, quarterly_df):
    df = quarterly_df if quarterly_df is not None and not quarterly_df.empty else annual_df
    basis = "quarterly" if quarterly_df is not None and not quarterly_df.empty else "annual"

    if df is None or df.empty:
        return {"error": "No cash flow data found"}

    operating_cf = parse_int(df_value(df, "Operating Cash Flow", 0))
    capex = parse_int(df_value(df, "Capital Expenditure", 0))
    capex_abs = abs(capex) if capex is not None else None
    fcf = operating_cf - capex_abs if operating_cf is not None and capex_abs is not None else None

    return {
        "period_end": str(df.columns[0].date()) if len(df.columns) > 0 else None,
        "basis": basis,
        "operating_cashflow": operating_cf,
        "capital_expenditures": capex,
        "free_cash_flow_estimate": fcf
    }

def summarize_earnings_dates(earnings_dates_df):
    if earnings_dates_df is None or earnings_dates_df.empty:
        return {"error": "No earnings dates data found"}

    try:
        latest_idx = earnings_dates_df.index[0]
        row = earnings_dates_df.iloc[0]

        return {
            "next_or_recent_earnings_date": str(latest_idx),
            "eps_estimate": parse_float(row.get("EPS Estimate")),
            "reported_eps": parse_float(row.get("Reported EPS")),
            "surprise_pct": parse_float(row.get("Surprise(%)"))
        }
    except:
        return {"error": "Could not parse earnings dates data"}

# =========================
# SERPAPI NEWS + SIMPLE SENTIMENT
# =========================
def get_news_results(query, num=10):
    data = safe_get(SERPAPI_BASE, {
        "engine": "google_news",
        "q": query,
        "api_key": SERPAPI_KEY
    })
    return data.get("news_results", [])[:num]

def simple_sentiment_score(text):
    positive_words = {
        "beat", "beats", "growth", "strong", "upside", "bullish", "surge", "record",
        "profit", "profits", "gain", "gains", "positive", "upgrade", "outperform",
        "expansion", "improves", "improved", "rebound", "momentum"
    }
    negative_words = {
        "miss", "misses", "weak", "downside", "bearish", "drop", "decline", "loss",
        "losses", "negative", "downgrade", "underperform", "risk", "lawsuit",
        "slowdown", "cuts", "cut", "pressure", "fall"
    }

    text = (text or "").lower()
    score = 0

    for w in positive_words:
        if w in text:
            score += 1
    for w in negative_words:
        if w in text:
            score -= 1

    return score

def summarize_news_sentiment(ticker, info=None):
    query = company_query_name(ticker, info)
    news = get_news_results(query, num=10)

    if not news:
        return {
            "query_used": query,
            "article_count": 0,
            "sentiment_score_total": 0,
            "sentiment_label": "no_data",
            "top_articles": []
        }

    scored_articles = []
    total_score = 0

    for item in news:
        title = item.get("title", "")
        snippet = item.get("snippet", "")
        source = item.get("source", {}).get("name") if isinstance(item.get("source"), dict) else item.get("source")
        date = item.get("date")
        link = item.get("link")

        combined = f"{title}. {snippet}"
        score = simple_sentiment_score(combined)
        total_score += score

        scored_articles.append({
            "title": title,
            "snippet": snippet,
            "source": source,
            "date": date,
            "link": link,
            "sentiment_score": score
        })

    if total_score > 2:
        label = "positive"
    elif total_score < -2:
        label = "negative"
    else:
        label = "mixed_or_neutral"

    return {
        "query_used": query,
        "article_count": len(scored_articles),
        "sentiment_score_total": total_score,
        "sentiment_label": label,
        "top_articles": scored_articles[:5]
    }


# =========================
# MAIN COLLECTION
# =========================
def collect_ticker_data(ticker):
    print(f"Collecting data for {ticker} with yfinance...")

    tkr = get_yf_ticker(ticker)

    info = get_info(tkr)
    price_hist = get_price_history(tkr, period="6mo", interval="1d")
    annual_income = get_income_statement(tkr)
    annual_balance = get_balance_sheet(tkr)
    annual_cashflow = get_cash_flow(tkr)
    quarterly_income = get_quarterly_income_statement(tkr)
    quarterly_balance = get_quarterly_balance_sheet(tkr)
    quarterly_cashflow = get_quarterly_cash_flow(tkr)
    earnings_dates = get_earnings_dates(tkr)

    time.sleep(SLEEP_BETWEEN_SERP_CALLS)
    news_summary = summarize_news_sentiment(ticker, info)

    result = {
        "ticker": ticker,
        "market": "SGX" if is_sgx_ticker(ticker) else "US/Other",
        "overview": summarize_overview(info),
        "price_summary": summarize_price_data(price_hist),
        "income_statement_summary": summarize_income_statement(annual_income, quarterly_income),
        "balance_sheet_summary": summarize_balance_sheet(annual_balance, quarterly_balance),
        "cash_flow_summary": summarize_cash_flow(annual_cashflow, quarterly_cashflow),
        "earnings_summary": summarize_earnings_dates(earnings_dates),
        "news_sentiment_summary": news_summary
    }

    return result


# =========================
# BUILD AI PROMPT
# =========================
def build_chatbot_prompt(result):
    prompt = f"""
You are a financial analysis assistant.

I will provide structured market, fundamentals, earnings, cash flow, balance sheet, price trend, and news sentiment data for one stock.

Your task:
1. Analyze the stock’s valuation, profitability, balance sheet strength, earnings quality, cash flow quality, and recent momentum.
2. Consider the news sentiment, but do not rely on it alone.
3. Identify major strengths and weaknesses.
4. Identify key risks and possible catalysts.
5. Provide a final view:
   - Recommendation: Buy / Hold / Avoid
   - Confidence: High / Medium / Low
   - Investment style fit: Value / Growth / Income / Speculative
   - Short thesis
6. If data is incomplete or weak, clearly explain what is missing.
7. Do not present the result as financial advice; present it as analytical opinion based only on the supplied data.

Here is the stock data:
{json.dumps(result, indent=2)}

Please return:
- A short executive summary
- A bullet-point analysis
- A final recommendation with rationale
- A risk section
- A catalyst section
"""
    return prompt.strip()


# =========================
# RUN
# =========================
result = collect_ticker_data(TICKER)

print("=" * 100)
print("STRUCTURED STOCK DATA")
print("=" * 100)
print(json.dumps(result, indent=2))

print("\n" + "=" * 100)
print("PROMPT TO PASTE INTO AI CHATBOT")
print("=" * 100)
print(build_chatbot_prompt(result))

STRUCTURED STOCK DATA
{
  "ticker": "GRAB",
  "market": "US/Other",
  "overview": {
    "symbol": "GRAB",
    "name": "Grab Holdings Limited",
    "exchange": "NMS",
    "currency": "USD",
    "country": "Singapore",
    "sector": "Technology",
    "industry": "Software - Application",
    "market_cap": 16114599936,
    "enterprise_value": 11604599808,
    "trailing_pe": 98.5,
    "forward_pe": 28.464094,
    "peg_ratio": 1.1,
    "price_to_book": 2.4733207,
    "book_value": 1.593,
    "dividend_rate": null,
    "dividend_yield": null,
    "eps_trailing": 0.04,
    "eps_forward": 0.13842,
    "revenue_ttm": 3552000000,
    "profit_margin": 0.106979996,
    "operating_margin": 0.02723,
    "return_on_assets": 0.00732,
    "return_on_equity": 0.04768,
    "target_mean_price": 5.89885,
    "beta": 0.875,
    "week_52_high": 6.62,
    "week_52_low": 3.18,
    "business_summary": "Grab Holdings Limited operates the Grab superapp in Cambodia, Indonesia, Malaysia, Myanmar, the Philippines, S